# CatBoost - Hyperparameter Sweeps

## Setup and Imports

In [1]:
import sys

sys.path.append("..")

import wandb
import dotenv

from src.api.run import sweep_catboost
from src.api.sweep import wandb_sweep

c:\Users\kybur\Repos\HSLU\aicomp\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
c:\Users\kybur\Repos\HSLU\aicomp\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because

In [2]:
dotenv.load_dotenv()
wandb.login()

wandb: Currently logged in as: v8-luky (aicomp-mmlm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Sweep Configuration

In [3]:
max_runs = 100
sweep_config = {
    "name": "CatBoost Hyperparameter Sweep",
    "method": "bayes",
    "metric": {"name": "cv_brier", "goal": "minimize"},
    "parameters": {
        "catboost_config": {
            "parameters": {
                "iterations": {"distribution": "int_uniform", "min": 200, "max": 1000},
                "learning_rate": {"distribution": "log_uniform_values", "min": 0.001, "max": 0.3},
                "depth": {"distribution": "int_uniform", "min": 3, "max": 10},
                "l2_leaf_reg": {"distribution": "log_uniform_values", "min": 0.1, "max": 10.0},
                "random_strength": {"distribution": "uniform", "min": 0.0, "max": 5.0},
                "bagging_temperature": {"distribution": "uniform", "min": 0.0, "max": 5.0},
                "subsample": {"distribution": "uniform", "min": 0.5, "max": 1.0},
                "border_count": {"values": [32, 64, 128, 254]},
                "grow_policy": {"values": ["SymmetricTree", "Lossguide", "Depthwise"]},
                "min_data_in_leaf": {"distribution": "int_uniform", "min": 1, "max": 10},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 2024},
                "start_season": {"value": 2003},
                "num_features": {"distribution": "int_uniform", "min": 4, "max": 100},
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_catboost, run_count=max_runs, project="aicomp-mmlm")

## Submission from Best Model

In [5]:
from src.dataloaders.simple import SeasonAverageDataLoader
from src.experiments import DefaultTracker
from src.models.catboost import CatBoostModel, CatBoostHyperparamConfig
from src.experiments.config import RunConfig
from src.submissions import generate_matchups, create_submission

In [6]:
run = wandb.Api().run("qvue3sg2")
config = run.config
config

{'run_config': {'num_features': 41,
  'start_season': 2003,
  'valid_season': 2024},
 'catboost_config': {'depth': 10,
  'subsample': 0.589985393320543,
  'iterations': 750,
  'grow_policy': 'SymmetricTree',
  'l2_leaf_reg': 7.0838302321173945,
  'border_count': 128,
  'learning_rate': 0.002106508973177542,
  'random_strength': 1.909815961032291,
  'min_data_in_leaf': 10,
  'bagging_temperature': 4.763414534326788}}

In [7]:
run_config = RunConfig(**config.get("run_config", {}))
dataloader = SeasonAverageDataLoader(run_config.num_features)
run_config

RunConfig(num_features=41, valid_season=2024, start_season=2003, data_loader='season_average')

In [8]:
hyperparameters = CatBoostHyperparamConfig(**config.get("catboost_config", {}))
hyperparameters

CatBoostHyperparamConfig(iterations=750, learning_rate=0.002106508973177542, depth=10, l2_leaf_reg=7.0838302321173945, random_strength=1.909815961032291, bagging_temperature=4.763414534326788, subsample=0.589985393320543, task_type='CPU', thread_count=-1, border_count=128, grow_policy='SymmetricTree', min_data_in_leaf=10, random_seed=42, verbose=0, allow_writing_files=False, loss_function='RMSE')

In [9]:
model = CatBoostModel(dataloader, hyperparameters, None, DefaultTracker({}))

In [10]:
season = 2025
create_submission(season=season, model=model, filename=f"submission_catboost_{season}.csv", fit=True)

metrics: {'train_brier': np.float64(0.14921858078167974)}, step: None


WindowsPath('C:/Users/kybur/Repos/HSLU/aicomp/code/submissions/submission_catboost_2025.csv')